# 0_test_access

Dieses Notebook testet einen ersten **Road-Erreichbarkeitsindikator** fuer `infraScan` auf dem externen Volume.

Hier werden bewusst **zwei Varianten von** `P_j` **parallel gefuehrt**:

1. `P_j` als **zielseitige Gesamtnachfrage** aus der OD-Matrix
2. `P_j` als **`pop + 0.5 * empl`** aus den Rasterdaten

Wichtige methodische Hinweise:

- Die Road-OD-Dateien enthalten aktuell **nur interzonale Beziehungen**.
- Intrazonale Beziehungen `i=i` werden im Code gezielt entfernt.
- Die `travel_time` in den Road-OD-Dateien sind **Stunden**, nicht Minuten.
- Fuer den Decay wird deshalb `beta = 3.0` pro Stunde verwendet, was `0.05` pro Minute entspricht.

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd

try:
    import rasterio
except ImportError as exc:
    raise ImportError(
        "Dieses Notebook braucht rasterio. Bitte im 'infraScan' oder 'infraScan2' conda env starten."
    ) from exc

DATA_ROOT = Path('/Volumes/WD_Windows/MSc_Thesis/data')
DATA_ROOT

## Verwendete Dateien

In [ ]:
paths = {
    'sq_od_tt': DATA_ROOT / 'infraScanRoad/traffic_flow/od/status_quo_od_tt.csv',
    'dev_od_tt': DATA_ROOT / 'infraScanRoad/traffic_flow/od/developments_od_tt.csv',
    'sq_source_raster': DATA_ROOT / 'infraScanRoad/Network/travel_time/source_id_raster.tif',
    'sq_access_raster': DATA_ROOT / 'infraScanRoad/Network/travel_time/travel_time_raster.tif',
    'dev_raster_dir': DATA_ROOT / 'infraScanRoad/Network/travel_time/developments',
    'pop_raster': DATA_ROOT / 'independent_variable/processed/pop20_corrected.tif',
    'empl_raster': DATA_ROOT / 'independent_variable/processed/empl20_corrected.tif',
}

pd.DataFrame(
    [{'key': k, 'path': str(v), 'exists': v.exists()} for k, v in paths.items()]
)

## OD-Dateien laden und Einheiten pruefen

In [ ]:
sq_tt = pd.read_csv(paths['sq_od_tt'])
dev_tt = pd.read_csv(paths['dev_od_tt'])

for df in (sq_tt, dev_tt):
    for col in ['origin', 'destination', 'demand', 'travel_time']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'development' in df.columns:
        df['development'] = pd.to_numeric(df['development'], errors='coerce')
    df['scenario'] = df['scenario'].astype(str)

sq_tt = sq_tt.dropna(subset=['origin', 'destination', 'demand', 'travel_time']).copy()
dev_tt = dev_tt.dropna(subset=['development', 'origin', 'destination', 'demand', 'travel_time']).copy()
sq_tt[['origin', 'destination']] = sq_tt[['origin', 'destination']].astype(int)
dev_tt[['development', 'origin', 'destination']] = dev_tt[['development', 'origin', 'destination']].astype(int)

display(sq_tt.head())
display(dev_tt.head())

summary = []
for scen in sorted(sq_tt['scenario'].unique()):
    g = sq_tt[sq_tt['scenario'] == scen]
    summary.append({
        'scenario': scen,
        'rows': len(g),
        'origins': g['origin'].nunique(),
        'destinations': g['destination'].nunique(),
        'min_tt': g['travel_time'].min(),
        'median_tt': g['travel_time'].median(),
        'max_tt': g['travel_time'].max(),
    })
pd.DataFrame(summary)

Interpretation:

- Die `travel_time`-Werte liegen groessenordnungsmassig bei etwa `1.1`, `1.6`, `2.9` usw.
- Das passt als **Stundenwerte** fuer den Korridor.
- Im `infraScanRoad`-Code wird die freie Link-Reisezeit ebenfalls explizit in **hours** berechnet.

## Intrazonale Beziehungen

In [ ]:
intrazonal_check = []
for scen in sorted(sq_tt['scenario'].unique()):
    g = sq_tt[sq_tt['scenario'] == scen]
    pairs = set(zip(g['origin'], g['destination']))
    origins = sorted(g['origin'].unique())
    missing_diagonal = [(i, i) for i in origins if (i, i) not in pairs]
    intrazonal_check.append({
        'scenario': scen,
        'n_pairs': len(pairs),
        'n_missing_diagonal': len(missing_diagonal),
        'sample_missing_diagonal': missing_diagonal[:5],
    })
pd.DataFrame(intrazonal_check)

## Raster laden und auf das Road-Raster ausrichten

Die Potenzialraster (`pop`, `empl`) liegen auf derselben Aufloesung, aber auf einem kleineren Ausschnitt.
Darum werden sie hier explizit in das Koordinatensystem und die Rasterform des Road-Catchment-Rasters eingesetzt.

In [ ]:
with rasterio.open(paths['sq_source_raster']) as src:
    sq_catch = src.read(1)
    road_transform = src.transform
    road_shape = (src.height, src.width)
    road_crs = src.crs

with rasterio.open(paths['sq_access_raster']) as src:
    sq_access_sec = src.read(1).astype(float)

with rasterio.open(paths['pop_raster']) as src:
    pop_small = src.read(1).astype(float)
    pop_transform = src.transform

with rasterio.open(paths['empl_raster']) as src:
    empl_small = src.read(1).astype(float)
    empl_transform = src.transform

print('road shape:', road_shape)
print('road crs:', road_crs)
print('road transform:', road_transform)
print('pop shape:', pop_small.shape)
print('empl shape:', empl_small.shape)

In [ ]:
def place_subset_on_road_grid(subset_arr: np.ndarray, subset_transform, road_shape, road_transform) -> np.ndarray:
    if road_transform.a != subset_transform.a or road_transform.e != subset_transform.e:
        raise ValueError('Raster resolutions differ; this helper expects identical cell size.')

    cell = road_transform.a
    row_off = int(round((road_transform.f - subset_transform.f) / cell))
    col_off = int(round((subset_transform.c - road_transform.c) / cell))

    out = np.zeros(road_shape, dtype=float)
    out[row_off:row_off + subset_arr.shape[0], col_off:col_off + subset_arr.shape[1]] = subset_arr
    return out


pop = place_subset_on_road_grid(pop_small, pop_transform, road_shape, road_transform)
empl = place_subset_on_road_grid(empl_small, empl_transform, road_shape, road_transform)

print('aligned pop total:', float(pop.sum()))
print('aligned empl total:', float(empl.sum()))

## Catchment-Statistiken

In [ ]:
def catchment_stats(catchment_raster, access_sec, pop_aligned, empl_aligned):
    valid = np.isfinite(catchment_raster) & np.isfinite(access_sec) & np.isfinite(pop_aligned) & np.isfinite(empl_aligned)
    valid &= catchment_raster > 0
    valid &= pop_aligned >= 0
    valid &= empl_aligned >= 0

    df = pd.DataFrame({
        'catchment': catchment_raster[valid].astype(int),
        'access_hr': access_sec[valid] / 3600.0,
        'pop': pop_aligned[valid],
        'empl': empl_aligned[valid],
    })

    grp = df.groupby('catchment', as_index=False).agg(
        pop=('pop', 'sum'),
        empl=('empl', 'sum'),
        access_hr=('access_hr', 'mean'),
    )
    grp['p_popempl'] = grp['pop'] + 0.5 * grp['empl']
    return grp


sq_stats = catchment_stats(sq_catch, sq_access_sec, pop, empl)
sq_stats.head()

## Accessibility-Funktionen

Hier wird die generalized travel time als

- `network travel time`
- `+ origin access`
- `+ destination access`

gebildet.

In [ ]:
BETA_PER_HOUR = 3.0


def compute_access(df: pd.DataFrame, potential_df: pd.DataFrame, time_col='gen_tt_hr', pot_col='Pj', beta=BETA_PER_HOUR):
    work = df.merge(potential_df[['destination', pot_col]], on='destination', how='left').copy()
    work[pot_col] = work[pot_col].fillna(0.0)
    work['contrib'] = np.exp(-beta * work[time_col]) * work[pot_col]
    return work.groupby('origin', as_index=False)['contrib'].sum().rename(columns={'contrib': 'access'})


def prepare_sq_generalized_tt(scenario: str):
    sq_s = sq_tt[sq_tt['scenario'] == scenario].copy()
    map_o = sq_stats[['catchment', 'access_hr']].rename(columns={'catchment': 'origin', 'access_hr': 'origin_access_hr'})
    map_d = sq_stats[['catchment', 'access_hr']].rename(columns={'catchment': 'destination', 'access_hr': 'dest_access_hr'})
    sq_s = sq_s.merge(map_o, on='origin', how='left').merge(map_d, on='destination', how='left')
    sq_s['gen_tt_hr'] = sq_s['travel_time'] + sq_s['origin_access_hr'].fillna(0.0) + sq_s['dest_access_hr'].fillna(0.0)
    return sq_s


def prepare_dev_generalized_tt(development: int, scenario: str):
    dev_s = dev_tt[(dev_tt['development'] == development) & (dev_tt['scenario'] == scenario)].copy()
    dev_source = paths['dev_raster_dir'] / f'dev{development}_source_id_raster.tif'
    dev_access = paths['dev_raster_dir'] / f'dev{development}_travel_time_raster.tif'

    with rasterio.open(dev_source) as src:
        dev_catch = src.read(1)
    with rasterio.open(dev_access) as src:
        dev_access_sec = src.read(1).astype(float)

    dev_stats = catchment_stats(dev_catch, dev_access_sec, pop, empl)
    map_o = dev_stats[['catchment', 'access_hr']].rename(columns={'catchment': 'origin', 'access_hr': 'origin_access_hr'})
    map_d = dev_stats[['catchment', 'access_hr']].rename(columns={'catchment': 'destination', 'access_hr': 'dest_access_hr'})
    dev_s = dev_s.merge(map_o, on='origin', how='left').merge(map_d, on='destination', how='left')
    dev_s['gen_tt_hr'] = dev_s['travel_time'] + dev_s['origin_access_hr'].fillna(0.0) + dev_s['dest_access_hr'].fillna(0.0)
    return dev_s, dev_stats


def delta_accessibility_for_variant(development: int, scenario: str, p_variant: str):
    sq_s = prepare_sq_generalized_tt(scenario)
    dev_s, dev_stats = prepare_dev_generalized_tt(development, scenario)

    if p_variant == 'demand_proxy':
        sq_p = sq_s.groupby('destination', as_index=False)['demand'].sum().rename(columns={'demand': 'Pj'})
        dev_p = dev_s.groupby('destination', as_index=False)['demand'].sum().rename(columns={'demand': 'Pj'})
    elif p_variant == 'pop_plus_0.5_empl':
        sq_p = sq_stats[['catchment', 'p_popempl']].rename(columns={'catchment': 'destination', 'p_popempl': 'Pj'})
        dev_p = dev_stats[['catchment', 'p_popempl']].rename(columns={'catchment': 'destination', 'p_popempl': 'Pj'})
    else:
        raise ValueError(f'Unknown p_variant: {p_variant}')

    sq_access = compute_access(sq_s, sq_p)
    dev_access = compute_access(dev_s, dev_p)

    out = sq_access.merge(dev_access, on='origin', suffixes=('_sq', '_dev'))
    out['delta_access'] = out['access_dev'] - out['access_sq']
    out['development'] = development
    out['scenario'] = scenario
    out['Pj_type'] = p_variant
    return out.sort_values('delta_access', ascending=False).reset_index(drop=True)

## Einzelergebnis fuer ein Beispiel

In [ ]:
example = delta_accessibility_for_variant(development=334, scenario='scenario_45', p_variant='pop_plus_0.5_empl')
example.head(15)

## Test fuer alle vorhandenen Developments und beide `P_j`-Varianten

In [ ]:
results = []
for scenario in sorted(sq_tt['scenario'].unique()):
    for development in sorted(dev_tt['development'].unique()):
        for p_variant in ['demand_proxy', 'pop_plus_0.5_empl']:
            out = delta_accessibility_for_variant(development=development, scenario=scenario, p_variant=p_variant)
            results.append({
                'scenario': scenario,
                'development': development,
                'Pj_type': p_variant,
                'sum_delta_access': out['delta_access'].sum(),
                'mean_delta_access': out['delta_access'].mean(),
                'max_delta_access': out['delta_access'].max(),
                'min_delta_access': out['delta_access'].min(),
            })

results_df = pd.DataFrame(results)
results_df.head()

In [ ]:
for scenario in sorted(results_df['scenario'].unique()):
    print('\nSCENARIO', scenario)
    for p_variant in ['demand_proxy', 'pop_plus_0.5_empl']:
        print('\nPj =', p_variant)
        display(
            results_df[
                (results_df['scenario'] == scenario) &
                (results_df['Pj_type'] == p_variant)
            ]
            .sort_values('sum_delta_access', ascending=False)
            .head(5)
        )

## Interpretation

Mit dem aktuellen Testaufbau gilt:

- `P_j = demand_proxy` ist naeher an eurer heutigen nachfragebasierten infraScan-Logik.
- `P_j = pop + 0.5 * empl` ist naeher am Accessibility-Paper.
- In beiden Varianten bleibt `development 334` im Test klar stark.
- Der absolute Zahlenwert ist zwischen den Varianten nicht direkt vergleichbar.
- Vergleichbar ist vor allem das **Ranking** und das **Vorzeichen der Delta-Accessibility**.

Wenn wir als Naechstes die intrazonalen Beziehungen einbeziehen wollten, braeuchten wir eine explizite Annahme fuer `t_ii` und fuer das Eigenpotenzial.

## Karten fuer die Top-5-Developments

Die Karten werden auf den Development-Voronoi-Polygonen dargestellt und mit der jeweiligen `delta_access` pro Ursprung eingefaerbt.

Default:

- `P_j = pop + 0.5 * empl`
- `scenario = scenario_45`

Das laesst sich unten leicht umstellen.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import mapclassify

MAP_SCENARIO = 'scenario_45'
MAP_PJ_TYPE = 'pop_plus_0.5_empl'
TOP_N = 5
MAP_OUTDIR = Path('plots/road_accessibility_maps')
MAP_OUTDIR.mkdir(parents=True, exist_ok=True)

statusquo_voronoi = gpd.read_file(DATA_ROOT / 'infraScanRoad/Network/travel_time/Voronoi_statusquo.gpkg')
statusquo_voronoi = statusquo_voronoi[['ID_point', 'geometry']].rename(columns={'ID_point': 'origin'})

statusquo_voronoi.head()

In [ ]:
top5 = (
    results_df[
        (results_df['scenario'] == MAP_SCENARIO) &
        (results_df['Pj_type'] == MAP_PJ_TYPE)
    ]
    .sort_values('sum_delta_access', ascending=False)
    .head(TOP_N)
)

top5

In [ ]:
def plot_development_accessibility_map(development: int, scenario: str, p_variant: str, output_dir: Path):
    delta_df = delta_accessibility_for_variant(development=development, scenario=scenario, p_variant=p_variant)
    vor_path = DATA_ROOT / 'infraScanRoad/Network/travel_time/developments' / f'dev{development}_Voronoi.gpkg'
    vor = gpd.read_file(vor_path)[['ID_point', 'geometry']].rename(columns={'ID_point': 'origin'})

    gdf = vor.merge(delta_df[['origin', 'delta_access']], on='origin', how='left')
    gdf['delta_access'] = gdf['delta_access'].fillna(0.0)

    vals = gdf['delta_access'].to_numpy()
    if np.allclose(vals, vals[0]):
        scheme = None
    else:
        try:
            scheme = mapclassify.FisherJenks(vals, k=min(5, len(np.unique(vals))))
        except Exception:
            scheme = mapclassify.Quantiles(vals, k=min(5, len(np.unique(vals))))

    fig, ax = plt.subplots(figsize=(8, 8), dpi=220)
    if scheme is None:
        gdf.plot(column='delta_access', cmap='RdYlGn', linewidth=0.15, edgecolor='white', legend=True, ax=ax)
    else:
        gdf.plot(
            column='delta_access',
            cmap='RdYlGn',
            scheme='UserDefined',
            classification_kwds={'bins': list(scheme.bins[:-1])},
            linewidth=0.15,
            edgecolor='white',
            legend=True,
            ax=ax,
        )

    ax.set_title(f'Road delta accessibility | dev {development} | {scenario} | {p_variant}', fontsize=11)
    ax.set_axis_off()
    plt.tight_layout()

    out = output_dir / f'road_access_dev{development}_{scenario}_{p_variant}.png'
    fig.savefig(out, bbox_inches='tight')
    plt.close(fig)
    return out


map_files = []
for dev in top5['development'].tolist():
    map_files.append(plot_development_accessibility_map(dev, MAP_SCENARIO, MAP_PJ_TYPE, MAP_OUTDIR))

pd.DataFrame({'map_file': [str(p) for p in map_files]})

Die PNGs liegen danach in `plots/road_accessibility_maps/`.

## Variante A: Rasterbasiertes Potenzial mit `TT_ij = A_i + N[a(i), e(j)] + E_j`

Diese Variante verwendet genau die von Laura gewuenschte Logik:

- `P_j = pop_j + 0.5 * empl_j` bleibt auf Rasterbasis
- `A_i` bleibt rasterbasiert
- `E_j` bleibt rasterbasiert
- nur `N[a(i), e(j)]` bleibt auf Catchment-/Access-Egress-Basis

Damit gilt fuer jede Ursprungszelle `i` und Zielzelle `j`:

`TT_ij = A_i + N[a(i), e(j)] + E_j`

und damit

`E_i = sum_j exp(-beta * TT_ij) * P_j`

Zur effizienten Berechnung wird die Zielseite komprimiert zu

`D_k = sum_{j in k} exp(-beta * E_j) * P_j`

wobei `k` ein Egress-/Zielcatchment ist. Dann folgt

`E_i = exp(-beta * A_i) * sum_k exp(-beta * N[a(i), k]) * D_k`

Das ist mathematisch aequivalent, solange `N` nur auf Catchment-/Egress-Ebene vorliegt.


In [ ]:
SCENARIO = 'scenario_45'
DEV_ID = 334

OUT_VARIANT_A = PLOTS_DIR / 'road_accessibility_rasterP_dev334'
OUT_VARIANT_A.mkdir(parents=True, exist_ok=True)

sq_tt_variant_a = sq_tt[sq_tt['scenario'] == SCENARIO][['origin', 'destination', 'travel_time']].copy()
dev_tt_variant_a = dev_tt[(dev_tt['scenario'] == SCENARIO) & (dev_tt['development'] == DEV_ID)][['origin', 'destination', 'travel_time']].copy()

def destination_weighted_potential(source_arr, access_hr_arr, potential_arr, beta_per_hour):
    valid = (source_arr > 0) & np.isfinite(access_hr_arr) & np.isfinite(potential_arr) & (potential_arr >= 0)
    frame = pd.DataFrame({
        'destination': source_arr[valid].astype(int),
        'weighted_potential': np.exp(-beta_per_hour * access_hr_arr[valid]) * potential_arr[valid],
    })
    return frame.groupby('destination', as_index=False)['weighted_potential'].sum().rename(columns={'weighted_potential': 'D_j'})

def accessibility_variant_a(tt_df, source_arr, access_hr_arr, potential_arr, beta_per_hour):
    dest_tbl = destination_weighted_potential(source_arr, access_hr_arr, potential_arr, beta_per_hour)
    d_map = dict(zip(dest_tbl['destination'].astype(int), dest_tbl['D_j']))

    cell_access = np.full(source_arr.shape, np.nan, dtype=float)
    origin_rows = []

    for origin_id in sorted(tt_df['origin'].astype(int).unique()):
        sub = tt_df[tt_df['origin'] == origin_id]
        if sub.empty:
            continue

        network_core = float(np.sum(
            np.exp(-beta_per_hour * sub['travel_time'].to_numpy())
            * sub['destination'].map(d_map).fillna(0.0).to_numpy()
        ))

        mask = (source_arr == origin_id) & np.isfinite(access_hr_arr)
        if not np.any(mask):
            continue

        values = np.exp(-beta_per_hour * access_hr_arr[mask]) * network_core
        cell_access[mask] = values

        origin_rows.append({
            'origin': int(origin_id),
            'accessibility_mean': float(values.mean()),
            'accessibility_min': float(values.min()),
            'accessibility_max': float(values.max()),
            'n_cells': int(values.size),
        })

    return cell_access, pd.DataFrame(origin_rows), dest_tbl

sq_grid_variant_a, sq_origin_variant_a, sq_dest_variant_a = accessibility_variant_a(
    tt_df=sq_tt_variant_a,
    source_arr=sq_source_arr,
    access_hr_arr=sq_access_hr,
    potential_arr=pot_arr,
    beta_per_hour=BETA_PER_HOUR,
)

dev_source_variant_a_path = ROAD_TRAVELTIME_DIR / 'developments' / f'dev{DEV_ID}_source_id_raster.tif'
dev_access_variant_a_path = ROAD_TRAVELTIME_DIR / 'developments' / f'dev{DEV_ID}_travel_time_raster.tif'

with rasterio.open(dev_source_variant_a_path) as src:
    dev_source_variant_a = src.read(1)
with rasterio.open(dev_access_variant_a_path) as src:
    dev_access_variant_a = src.read(1).astype(float) / 3600.0

dev_grid_variant_a, dev_origin_variant_a, dev_dest_variant_a = accessibility_variant_a(
    tt_df=dev_tt_variant_a,
    source_arr=dev_source_variant_a,
    access_hr_arr=dev_access_variant_a,
    potential_arr=pot_arr,
    beta_per_hour=BETA_PER_HOUR,
)

delta_grid_variant_a = dev_grid_variant_a - sq_grid_variant_a

sq_origin_variant_a.to_csv(OUT_VARIANT_A / 'sq_origin_values_rasterP.csv', index=False)
dev_origin_variant_a.to_csv(OUT_VARIANT_A / 'dev334_origin_values_rasterP.csv', index=False)
sq_dest_variant_a.to_csv(OUT_VARIANT_A / 'sq_destination_weighted_potential_rasterP.csv', index=False)
dev_dest_variant_a.to_csv(OUT_VARIANT_A / 'dev334_destination_weighted_potential_rasterP.csv', index=False)

pd.DataFrame([{
    'scenario': SCENARIO,
    'development': DEV_ID,
    'sq_grid_mean': float(np.nanmean(sq_grid_variant_a)),
    'dev_grid_mean': float(np.nanmean(dev_grid_variant_a)),
    'delta_grid_sum': float(np.nansum(delta_grid_variant_a)),
    'delta_grid_min': float(np.nanmin(delta_grid_variant_a)),
    'delta_grid_max': float(np.nanmax(delta_grid_variant_a)),
    'negative_cell_share': float(np.nanmean(delta_grid_variant_a < 0)),
    'positive_cell_share': float(np.nanmean(delta_grid_variant_a > 0)),
}]).to_csv(OUT_VARIANT_A / 'summary.csv', index=False)

display(sq_origin_variant_a.head())
display(dev_origin_variant_a.head())
display(pd.read_csv(OUT_VARIANT_A / 'summary.csv'))
